In [1]:
import os
from uuid_utils import uuid7

import numpy as np
import pandas as pd
from clickhouse_connect import get_client
from dotenv import load_dotenv
import pandera as pa
import polars as pl
from enum import Enum, IntEnum

import datetime as dt

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)


# Parameters

In [2]:
file_directory = "../data/bronze/depdev/"
file = "PSGC-3Q-2025-Publication-Datafile.xlsx"
valid_from = "2025-10-13T00:00:00.000+0800"

In [ ]:
SettlementTypeEnum = pl.Enum(categories=["urban", "rural", "-"])
BarangayStatusEnum = pl.Enum(categories=["poblacion", "capital"])


# Main Process

In [5]:
df = pl.read_excel(source=file_directory + file, sheet_name="PSGC")
df.sample(10)

Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string


10-digit PSGC,Name,Correspondence Code,Geographic Level,Old names,City Class,Income Classification (DOF DO No. 074.2024),Urban / Rural (based on 2020 CPH),2024 Population,__UNNAMED__9,Status
str,str,i64,str,str,str,str,str,i64,str,str
"""1400101011""","""Lingtan""",140101011,"""Bgy""",null,null,null,"""R""",1037,null,null
"""0500509015""","""San Jose""",50509015,"""Bgy""",null,null,null,"""U""",5352,null,null
"""0701234016""","""Pamacsalan""",71234016,"""Bgy""",null,null,null,"""R""",796,null,null
"""0306905045""","""Santiago""",36905045,"""Bgy""",null,null,null,"""U""",6147,null,null
"""0201508003""","""Cabaritan""",21508003,"""Bgy""",null,null,null,"""R""",873,null,null
"""1903617017""","""Boto Ambolong""",153617017,"""Bgy""",null,null,null,"""R""",2031,null,null
"""1404401012""","""Lias Silangan""",144401012,"""Bgy""",null,null,null,"""R""",366,null,null
"""0500504049""","""Binogsacan Upper""",50504049,"""Bgy""",null,null,null,"""R""",1410,null,null
"""0806025021""","""Ranera""",86025021,"""Bgy""",null,null,null,"""R""",368,null,null


In [6]:
# check columns
df.columns

['10-digit PSGC',
 'Name',
 'Correspondence Code',
 'Geographic Level',
 'Old names',
 'City Class',
 'Income\r\nClassification (DOF DO No. 074.2024)',
 'Urban / Rural\r\n(based on 2020 CPH)',
 '2024 Population',
 '__UNNAMED__9',
 'Status']

In [7]:
# build corrected column names
correct_columns = {
    "10-digit PSGC": "psgc_id",
    "Name": "psgc_name",
    "Correspondence Code": "correspondence_code",
    "Geographic Level": "geographic_level",
    "Old names": "old_name",
    "City Class": "city_class",
    "Income\r\nClassification (DOF DO No. 074.2024)": "income_classification",
    "Urban / Rural\r\n(based on 2020 CPH)": "settlement_type",
    "2024 Population": "population",
    "__UNNAMED__9": "remarks",
    "Status": "status",
}

In [8]:
renamed_df = df.rename(correct_columns)
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,i64,str,str,str,str,str,i64,str,str
"""0701219007""","""Poblacion""",71219007,"""Bgy""",null,null,null,"""R""",2485,null,null
"""0701207000""","""Batuan""",71207000,"""Mun""",null,null,"""4th""","""""",14423,null,null
"""1804510004""","""Cabanbanan""",64510004,"""Bgy""",null,null,null,"""R""",2798,null,null
"""0304930026""","""Homestead I""",34930026,"""Bgy""",null,null,null,"""R""",2049,null,null
"""1001803007""","""Poblacion""",101803007,"""Bgy""",null,null,null,"""U""",2937,null,null
"""0103307026""","""Payocpoc Norte Este""",13307026,"""Bgy""",null,null,null,"""R""",1266,null,null
"""1001315008""","""Dologon""",101315008,"""Bgy""",null,null,null,"""U""",14058,null,null
"""0304918024""","""South Central Poblacion""",34918024,"""Bgy""",null,null,null,"""R""",549,null,null
"""0105542010""","""Macaycayawan""",15542010,"""Bgy""",null,null,null,"""R""",736,null,null


# col: `psgc_id`

In [9]:
# verify that all psgc_id have length 10
assert len(renamed_df["psgc_id"].str.len_chars().value_counts()["psgc_id"])==1
assert renamed_df["psgc_id"].str.len_chars().value_counts().row(0)[0] == 10

renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,i64,str,str,str,str,str,i64,str,str
"""1102324007""","""Mamangan""",112324007,"""Bgy""",null,null,null,"""R""",2739,null,null
"""0203107013""","""Macalaoat""",23107013,"""Bgy""",null,null,null,"""R""",1384,null,null
"""0102809013""","""Guerrero """,12809013,"""Bgy""",null,null,null,"""R""",1531,null,"""Pob."""
"""0803713047""","""Paula""",83713047,"""Bgy""",null,null,null,"""R""",321,null,null
"""1004203026""","""Rufino Lumapas""",104203026,"""Bgy""",null,null,null,"""R""",1005,null,null
"""0501606006""","""Bagong Silang II""",51606006,"""Bgy""",null,null,null,"""R""",1726,null,null
"""1208007031""","""Upper Mainit""",128007031,"""Bgy""",null,null,null,"""R""",4193,null,null
"""1900713004""","""Bukut-Umus""",150713004,"""Bgy""",null,null,null,"""R""",5733,null,null
"""0907308007""","""Datu Totocan""",97308007,"""Bgy""",null,null,null,"""R""",955,null,null


# col: `correspondence_code`

In [10]:
renamed_df = renamed_df.with_columns(
    pl.col("correspondence_code").cast(pl.Utf8)
)

In [11]:
renamed_df = renamed_df.with_columns(pl.col("correspondence_code").fill_null(""))
renamed_df = renamed_df.with_columns(
    pl.when(pl.col("correspondence_code").str.len_chars() == 8)
    .then(pl.col("correspondence_code").str.zfill(9))
    .otherwise(pl.col("correspondence_code")),
)

In [12]:
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,str,i64,str,str
"""0203134012""","""Poblacion 1""","""023134012""","""Bgy""",null,null,null,"""R""",1532,null,null
"""0203119008""","""Olango""","""023119008""","""Bgy""",null,null,null,"""R""",2492,null,null
"""0701243036""","""Tanghaligue""","""071243036""","""Bgy""",null,null,null,"""R""",2362,null,null
"""0806416014""","""Tuba-on""","""086416014""","""Bgy""",null,null,null,"""R""",526,null,null
"""0806003052""","""Dawo""","""086003052""","""Bgy""",null,null,null,"""R""",1307,null,null
"""0506203019""","""Daganas""","""056203019""","""Bgy""",null,null,null,"""R""",446,null,null
"""0401028016""","""San Luis""","""041028016""","""Bgy""",null,null,null,"""U""",2997,null,null
"""0803701024""","""Dingle""","""083701024""","""Bgy""",null,null,null,"""R""",549,null,null
"""1606707006""","""Cambas-ac""","""166707006""","""Bgy""",null,null,null,"""R""",786,null,null


# col: `settlement_type`

In [13]:
settlement_type_map = {"R": "rural", "U": "urban", "": None}
renamed_df = renamed_df.with_columns(
    pl.col("settlement_type").replace(settlement_type_map)
)
renamed_df = renamed_df.with_columns(pl.col("settlement_type").cast(SettlementTypeEnum))
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,str
"""0806408018""","""San Joaquin""","""086408018""","""Bgy""",null,null,null,"""rural""",1191,null,null
"""0603017009""","""Batuan""","""063017009""","""Bgy""",null,null,null,"""rural""",1555,null,null
"""0603023057""","""San Julian ""","""063023057""","""Bgy""",null,null,null,"""rural""",1439,null,"""Pob."""
"""0500508024""","""Francia""","""050508024""","""Bgy""",null,null,null,"""rural""",862,null,null
"""0603002022""","""Cunsad""","""063002022""","""Bgy""",null,null,null,"""rural""",378,null,null
"""0702249019""","""Villahermosa""","""072249019""","""Bgy""",null,null,null,"""rural""",621,null,null
"""1001322014""","""Mauswagon""","""101322014""","""Bgy""",null,null,null,"""rural""",1358,null,null
"""1102414013""","""Mckinley""","""112414013""","""Bgy""",null,null,null,"""rural""",2110,null,null
"""1908705010""","""Capiton""","""153807010""","""Bgy""",null,null,null,"""urban""",5756,null,null


In [14]:
renamed_df["status"].value_counts()

status,count
str,u32
"""Pob.""",2773
"""Capital""",82
null,40914


In [15]:
barangay_status_map = {
    "Pob.": "poblacion",
    "Capital": "capital",
}
renamed_df = renamed_df.with_columns(pl.col("status").replace(barangay_status_map))
renamed_df = renamed_df.with_columns(pl.col("status").cast(BarangayStatusEnum))
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,enum
"""0504107003""","""Buyo""","""054107003""","""Bgy""",null,null,null,"""rural""",1740,null,null
"""0701220012""","""Cambacol""","""071220012""","""Bgy""",null,null,null,"""rural""",251,null,null
"""0600606002""","""Bagacay""","""060606002""","""Bgy""",null,null,null,"""rural""",1679,null,null
"""0405649009""","""Caigdal""","""045649009""","""Bgy""",null,null,null,"""rural""",672,null,null
"""1903622011""","""Cadayonan""","""153622011""","""Bgy""",null,null,null,"""rural""",1625,null,null
"""0102905027""","""Sabang""","""012905027""","""Bgy""",null,null,null,"""rural""",1534,null,null
"""0306903051""","""Poblacion I""","""036903051""","""Bgy""",null,null,null,"""rural""",1079,null,null
"""0907333001""","""Baking""","""097333001""","""Bgy""",null,null,null,"""rural""",1597,null,null
"""1908821010""","""Linantangan""","""153837009""","""Bgy""",null,null,null,"""rural""",2508,null,null


# col: `old_name`, `remarks`

In [16]:
renamed_df = renamed_df.with_columns(
    [
        pl.col("old_name").fill_null(""),
        pl.col("remarks").fill_null(""),
    ]
)
renamed_df.sample(5)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,enum
"""0906609021""","""Laum Buwahan""","""156609021""","""Bgy""","""""",null,null,"""rural""",2005,"""""",null
"""1004204004""","""Dapacan Alto""","""104204004""","""Bgy""","""""",null,null,"""rural""",1190,"""""",null
"""1606716000""","""Pilar""","""166716000""","""Mun""","""""",null,"""4th""",null,11135,"""""",null
"""1705103005""","""Bulacan""","""175103005""","""Bgy""","""""",null,null,"""rural""",727,"""""",null
"""0804810018""","""Imelda""","""084810018""","""Bgy""","""""",null,null,"""rural""",249,"""""",null


# filter: `brgy` only

In [17]:
renamed_df = renamed_df.filter(pl.col("geographic_level")=="Bgy")

In [18]:
renamed_df = renamed_df.drop(
    [
        "city_class",
        "income_classification",
        "population",
        "geographic_level",
    ],
    strict=False
)

In [19]:
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,old_name,settlement_type,remarks,status
str,str,str,str,enum,str,enum
"""1705304002""","""Bugsuk""","""175304002""","""New Cagayancillo""","""urban""","""""",null
"""0702213019""","""Tagnucan""","""072213019""","""""","""rural""","""""",null
"""1030500048""","""F. S. Catanico""","""104305048""","""""","""rural""","""""",null
"""0403409002""","""San Antonio""","""043409002""","""""","""urban""","""""",null
"""0304911034""","""Macatcatuit""","""034911034""","""""","""rural""","""""",null
"""0802605007""","""Can-ilay""","""082605007""","""""","""rural""","""""",null
"""0102909002""","""Baracbac""","""012909002""","""""","""rural""","""""",null
"""0600602017""","""Igtunarum""","""060602017""","""""","""rural""","""""",null
"""1606819002""","""Bagong Lungsod ""","""166819002""","""""","""urban""","""""","""poblacion"""


In [20]:
import blake3

renamed_df = renamed_df.with_columns(
    pl.struct(["psgc_name", "psgc_id"])
    .map_elements(
        lambda row: blake3.blake3(
            f"{row["psgc_name"]}_{row["psgc_id"]}".encode(encoding="utf-8")
        ).hexdigest()
    )
    .alias("identity_hash"),
    pl.struct(
        ["settlement_type", "status", "remarks", "correspondence_code", "old_name"]
    )
    .map_elements(
        lambda row: blake3.blake3(
            (
                f"{row["settlement_type"]}_{row["status"]}_{row["remarks"]}"
                + f"_{row["correspondence_code"]}_{row["old_name"]}"
            ).encode(encoding="utf-8")
        ).hexdigest()
    )
    .alias("fields_hash"),
)

In [60]:
renamed_df.with_columns(
    pl.struct(
        ["settlement_type", "status", "remarks", "correspondence_code", "old_name"]
    ).map_elements(
        lambda row:
            
                f"{row["settlement_type"]}_{row["status"]}_{row["remarks"]}"
                + f"_{row["correspondence_code"]}_{row["old_name"]}"
            
        
    ).alias("field_to_hash")
)[0]["field_to_hash"][0]

'urban_None__137501001_'

In [21]:
# replace logic here
from uuid import uuid4

renamed_df = renamed_df.with_columns(
    pl.Series(
        name="surrogate_id", values=[str(uuid4()) for _ in range(len(renamed_df))]
    ).cast(pl.String).alias("surrogate_id")
)

In [39]:
release_date = pl.Series([valid_from]).str.strptime(
    pl.Datetime, "%Y-%m-%dT%H:%M:%S%.3f%z"
)[0]

perpetual = pl.Series(["9999-12-31T23:59:59.999+08:00"]).str.strptime(
    pl.Datetime, "%Y-%m-%dT%H:%M:%S%.3f%z"
)[0]

In [40]:

renamed_df = renamed_df.with_columns(
    pl.lit(release_date).alias("valid_from"),
    pl.lit(perpetual).alias("valid_to").cast(pl.Datetime(time_zone="")),
)

In [41]:
renamed_df = renamed_df.with_columns(
    pl.lit(dt.datetime.now(tz=dt.timezone.utc)).alias("ingestion_datetime")
)

In [42]:
# ordering columns
barangay = renamed_df.select(
    [   
        "surrogate_id",
        "ingestion_datetime",
        "psgc_id",
        "psgc_name",
        "correspondence_code",
        "old_name",
        "settlement_type",
        "status",
        "remarks",
        "identity_hash",
        "fields_hash",
        "valid_from",
        "valid_to",
    ]
)

In [43]:
barangay.filter(pl.col("status")=="")

surrogate_id,ingestion_datetime,psgc_id,psgc_name,correspondence_code,old_name,settlement_type,status,remarks,identity_hash,fields_hash,valid_from,valid_to
str,"datetime[μs, UTC]",str,str,str,str,enum,enum,str,str,str,"datetime[μs, UTC]",datetime[μs]


# Applying Kimball's SCD Type 2

```

if identity hash is same and fields hash is same:
    do nothing
if identity hash is same and fields hash is different:
    then fields has an update
if identity_hash is different and fields hash is the same:
    then identity has an update. Triage manually
if identity_hash is different and fields is different:
    likely a new record
        run with barangay to determine similarity and triage manually
        
```

In [ ]:



os.environ["CLICKHOUSE_USE_TIMEZONE"] = "Asia/Manila"
load_dotenv()
import datetime as dt




try:
    DimBarangay.metadata.tables[DimBarangay.__tablename__].drop(bind=engine)
except DatabaseException as e:
    print(f"table not found: {DimBarangay.__tablename__}")
DimBarangay.metadata.tables["dim_barangay"].create(bind=engine) 

In [45]:
barangay = barangay.with_columns(
    pl.lit(dt.datetime.now(tz=dt.timezone.utc)).alias("ingestion_datetime")
)

In [33]:
raise KeyboardInterrupt()

KeyboardInterrupt: 

In [46]:
for i in range(0, len(barangay), 5000):
    chunk = barangay[i:i+5000]
    client.insert_arrow(table=DimBarangay.__tablename__, arrow_table=chunk.to_arrow())

In [ ]:
chunk["status"].to_arrow()

In [ ]:
len(barangay.columns)

In [49]:
f"{barangay[0]["status"]}"

"shape: (1,)\nSeries: 'status' [enum]\n[\n\tnull\n]"

In [ ]:
import pandera.polars as pa
from typing import Dict
from pandera.engines.polars_engine import DateTime, String, UInt32, Enum

psgc_id_length_check = pa.Check(lambda s: len(s) == 10 or len(s) == 0)
legacy_psgc_id_length_check = pa.Check(lambda s: len(s) == 9 or len(s) == 0)
uuid_length = pa.Check(lambda s: len(s) == 36)
blake3_hash_len = pa.Check(lambda s: len(s) == 64)

schema_dim_barangay: Dict[str, pa.Column] = {
    "surrogate_id": pa.Column(dtype=String, checks=[uuid_length]),
    "psgc_id": pa.Column(dtype=String, checks=[psgc_id_length_check]),
    "psgc_name": pa.Column(dtype=String),
    "correspondence_code": pa.Column(
        dtype=String, checks=[legacy_psgc_id_length_check]
    ),
    "old_name": pa.Column(dtype=String, checks=[]),
    "settlement_type": pa.Column(dtype=SettlementTypeEnum, checks=[]),
    "status": pa.Column(dtype=BarangayStatusEnum, checks=[]),
    "remarks": pa.Column(dtype=String, checks=[]),
    "valid_from": pa.Column(dtype=DateTime, checks=[]),
    "valid_to": pa.Column(dtype=DateTime, checks=[], nullable=True),
    "identity_hash": pa.Column(dtype=String, checks=[blake3_hash_len]),
    "fields_hash": pa.Column(dtype=String, checks=[blake3_hash_len]),
}
barangay_schema = pa.DataFrameSchema(columns=schema_dim_barangay)

In [ ]:
barangay_schema.validate(barangay)